In [ ]:
# ============================================================================
# 0. Setup: зависимости и расположение данных
# ============================================================================
"""
Готовит окружение и определяет пути к данным и модели.

В Colab устанавливает зависимости при первом запуске. Локально:
    pip install -r requirements.txt

Определяет корень репозитория по маркеру data/raw/ruscorpora_content.csv:
  - если ноутбук открыт из клонированного репозитория — находит его
    вверх по дереву от текущей рабочей директории;
  - если папки нет (первый запуск в чистом Colab) — клонирует
    репозиторий с GitHub в /content/<REPO_NAME>.

Экспортирует в глобальную область:
  REPO, DATA_DIR, RAW_DIR, RESULTS_DIR, MODEL_DIR
"""

import os
import subprocess
import warnings
from pathlib import Path

import pandas as pd

# Colab выводит DeprecationWarning из jupyter_client; к проекту не относится.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*utcnow.*")


# --- 1. Зависимости (только в Colab) ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "setfit", "transformers", "datasets", "scikit-learn",
        "huggingface_hub",
    ])


# --- 2. Расположение репозитория ---
REPO_URL  = "https://github.com/bursansan/dacha-langclassifier.git"
REPO_NAME = "dacha-langclassifier"
MARKER    = Path("data") / "raw" / "ruscorpora_content.csv"


def locate_repo_root(start=None):
    """Ищет корень репо вверх по дереву по маркеру."""
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / MARKER).exists():
            return p
    return None


REPO = locate_repo_root()

if REPO is None:
    candidate = Path("/content") / REPO_NAME
    if not (candidate / MARKER).exists():
        print(f"Клонирую репозиторий: {REPO_URL}")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(candidate)],
            check=True,
        )
    REPO = candidate

os.chdir(REPO)

DATA_DIR    = Path("data")
RAW_DIR     = DATA_DIR / "raw"
RESULTS_DIR = Path("results")
MODEL_DIR   = Path("model") / "setfit_dacha_classifier"

print(f"REPO:        {REPO}")
print(f"DATA_DIR:    {DATA_DIR.resolve()}")
print(f"RAW_DIR:     {RAW_DIR.resolve()}")
print(f"RESULTS_DIR: {RESULTS_DIR.resolve()}")


# --- 3. Проверка ожидаемых файлов ---
EXPECTED = [
    RAW_DIR / "ruscorpora_content.csv",
    DATA_DIR / "annotated_corpus.csv",
    DATA_DIR / "train_house.csv",
    DATA_DIR / "train_other.csv",
    DATA_DIR / "valid.csv",
    DATA_DIR / "test.csv",
]

missing = [p for p in EXPECTED if not p.exists()]
for p in EXPECTED:
    print(f"  {p.name}: {'✓' if p.exists() else '✗'}")
if missing:
    raise FileNotFoundError(
        f"Не найдены файлы: {[str(p) for p in missing]} в {REPO}"
    )


# --- 4. Сводка по корпусу ---
ann = pd.read_csv(DATA_DIR / "annotated_corpus.csv",
                  sep=";", encoding="utf-8-sig", dtype=str)
assert "reference" in ann.columns, \
    "annotated_corpus.csv должен содержать колонку reference (HOUSE/OTHER)"
assert ann["reference"].isin(["HOUSE", "OTHER"]).all(), \
    "reference должен содержать только HOUSE/OTHER"

print(f"\nКорпус: {len(ann)} вхождений слова «дача»")
print(f"  HOUSE: {(ann.reference=='HOUSE').sum()}")
print(f"  OTHER: {(ann.reference=='OTHER').sum()}")

In [ ]:
# ============================================================================
# 1. Подготовка данных: train / valid / test
# ============================================================================
"""
Готовит train / valid / test для бинарного классификатора слова «дача».

Дано: 703 вхождения слова «дача» из корпуса НКРЯ с reference-разметкой
HOUSE / OTHER (см. README, раздел Data)
train и valid строятся из синтетических шаблонов

Классы:
  HOUSE (0) — «дача» как загородный дом, участок или строение.
  OTHER (1) — «дача» как передача / вручение (показаний, взятки, розог и т.д.).

Вход:  annotated_corpus.csv
Выход: train_house.csv, train_other.csv, valid.csv, test.csv

Принципы:
  1. 703 размеченных вхождения — только test. Ни один train-пример не
     должен совпадать с тестом по нормализованному тексту, Jaccard ≥ 0.7 или
     подстроке (≥5 слов).
  2. Test-строки обрезаны до окна ±100 символов вокруг целевого слова,
     чтобы выровнять длины train и test.
  3. Train — синтетика (шаблоны с подстановкой падежа, формулы, длинные
     парафразы), а не копии тестовых строк.
  4. Oversampling на этапе data prep не применяется: соотношение классов
     в train задаётся синтетическими шаблонами (≈2:1). Балансировка,
     если нужна, делается на этапе обучения.
  5. Выходные файлы следуют проектной конвенции кодирования меток:
     эталон — колонка reference со значениями HOUSE/OTHER;
     обучающие сплиты — колонка label со значениями 0/1.
"""

from pathlib import Path
import pandas as pd
import numpy as np
import re
import random

# DATA_DIR определяется в ячейке 0 (Setup). Если ячейка запускается
# изолированно — находим data/ автоматически, поднимаясь вверх от CWD.
if "DATA_DIR" not in globals():
    for _p in (Path.cwd(), *Path.cwd().parents):
        if (_p / "data" / "raw" / "ruscorpora_content.csv").exists():
            DATA_DIR = _p / "data"
            break
    else:
        raise FileNotFoundError(
            "Не найдена папка data/. Запустите ячейку 0 (Setup) "
            "или откройте ноутбук из корня репозитория."
        )
print(f"DATA_DIR = {DATA_DIR}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


# ---------------------------------------------------------------------------
# Утилиты нормализации
# ---------------------------------------------------------------------------
def norm_soft(s):
    """Нижний регистр + схлопывание пробелов. Для substring-проверок."""
    return re.sub(r"\s+", " ", str(s).lower().strip())


def norm_hard(s):
    """Пунктуация → пробел, затем схлопывание. Для exact-match."""
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", str(s).lower()).strip())


def nset(s):
    """Множество слов. Для Jaccard."""
    return set(re.sub(r"[^\w\s]", " ", str(s).lower()).split())


def jac_set(A, B):
    return len(A & B) / len(A | B) if (A | B) else 0.0


# ---------------------------------------------------------------------------
# 1. Test: 703 размеченных вхождения корпуса
# ---------------------------------------------------------------------------
print("=" * 60)
print("1. ПОДГОТОВКА ТЕСТА")

test_full = pd.read_csv(DATA_DIR / "annotated_corpus.csv",
                        sep=";", encoding="utf-8-sig", dtype=str)

assert "reference" in test_full.columns, (
    "annotated_corpus.csv должен содержать колонку reference (HOUSE/OTHER). "
    "Если файл ещё в старом формате (label: 0/1) — переименуйте колонку "
    "и значения вручную один раз."
)
assert test_full["reference"].isin(["HOUSE", "OTHER"]).all(), (
    "reference должен содержать только HOUSE/OTHER"
)

print(f"  Разметка: HOUSE={(test_full.reference=='HOUSE').sum()}, "
      f"OTHER={(test_full.reference=='OTHER').sum()}")

# Окно ±100 символов вокруг Center — выравнивает длины train и test.
WINDOW_CHARS = 100


def make_window(row, pad=WINDOW_CHARS):
    fc = str(row["Full context"])
    center = str(row["Center"]) if pd.notna(row["Center"]) else ""
    if not center or center not in fc:
        return fc[:2 * pad].strip()
    pos = fc.find(center)
    start = max(0, pos - pad)
    end = min(len(fc), pos + len(center) + pad)
    out = fc[start:end].strip()
    if start > 0:
        out = "…" + out
    if end < len(fc):
        out = out + "…"
    return out


test_full["text"] = test_full.apply(make_window, axis=1).str.strip()
test_full = test_full[test_full["text"].str.len() > 0].reset_index(drop=True)
print(f"  Медиана длины окна: {int(test_full['text'].str.len().median())} символов")

test_df = test_full[["text", "reference"]].copy()
test_hard_set = set(norm_hard(t) for t in test_df["text"])
test_soft_list = [norm_soft(t) for t in test_df["text"]]
test_sets = [nset(t) for t in test_df["text"]]

print("\n  Распределение по авторам (топ-10):")
by_author = test_full.groupby("Author", dropna=False)["reference"].agg(
    n="count",
    house=lambda x: (x == "HOUSE").sum(),
    other=lambda x: (x == "OTHER").sum(),
).sort_values("n", ascending=False)
print(by_author.head(10).to_string())


# ---------------------------------------------------------------------------
# 2. Train: HOUSE
# ---------------------------------------------------------------------------
house_pairs = [
    ("Мы поехали на {d} в выходные", ["дачу"]),
    ("Он приезжает на {d} только летом", ["дачу"]),
    ("Перевез на {d} книги и мебель", ["дачу"]),
    ("Приглашал на свою {d} за город", ["дачу"]),
    ("Они снимали {d} на всё лето", ["дачу"]),
    ("Провести лето на {d}", ["даче"]),
    ("Жить на {d} круглый год", ["даче"]),
    ("На {d} всегда много работы", ["даче"]),
    ("На {d} растут яблони и смородина", ["даче", "дачах"]),
    ("Он оборудовал мастерскую на {d}", ["даче"]),
    ("Вернулись с {d} поздно вечером", ["дачи"]),
    ("Привезли с {d} варенье и яблоки", ["дачи"]),
    ("Дорога с {d} занимает два часа", ["дачи"]),
    ("Тропинка вела к {d}", ["даче", "дачам"]),
    ("К {d} пристроили веранду", ["даче"]),
    ("Стояли у {d} и разговаривали", ["дачи", "дач"]),
    ("Проезжали мимо {d} два раза", ["дачи", "дач"]),
    ("Остановились около {d}", ["дачи", "дач"]),
    ("{d} Крэнстонов", ["дача", "дачу", "дачи", "дач"]),
    ("{d} Сталина", ["дача", "дачу", "дачи"]),
    ("{d} родителей", ["дача", "дачу", "дачи"]),
    ("{d} деда", ["дача", "дачу", "дачи"]),
    ("{d} соседа", ["дача", "дачу", "дачи"]),
    ("Владелец {d} приехал на выходные", ["дачи"]),
    ("Хозяин {d} продал участок", ["дачи"]),
    ("Сосед по {d} оказался врачом", ["даче"]),
    ("Последняя {d} стояла у леса", ["дача"]),
    ("Пустая {d} выглядела заброшенной", ["дача"]),
    ("Государственные {d} сохраняли за семьями", ["дачи", "дач"]),
    ("Ведомственные {d} распределяли по решению", ["дачи", "дач"]),
    ("Казённые {d} числились на балансе", ["дачи", "дач"]),
    ("Академические {d} давали профессуре", ["дачи", "дач"]),
    ("Большим чинам такие {d} давали", ["дачи", "дач"]),
    ("Призывники строили генералам {d}", ["дачи", "дач"]),
    ("Землю раздали под {d}", ["дачи"]),
    ("На балансе института несколько {d}", ["дач"]),
    ("Продали {d} и переехали в город", ["дачу"]),
    ("Купили {d} в садовом товариществе", ["дачу"]),
    ("Построили {d} своими руками", ["дачу"]),
    ("Затопили {d} и сели пить чай", ["дачу"]),
    ("Обнесли {d} высоким забором", ["дачу"]),
    ("Он продал {d} и купил квартиру", ["дачу"]),
    ("Они арендовали {d} у знакомых", ["дачу"]),
    ("Старая {d} покосилась от времени", ["дача"]),
    ("Новая {d} стояла у самого леса", ["дача"]),
    ("Деревянная {d} сгорела зимой", ["дача"]),
    ("Заброшенная {d} заросла бурьяном", ["дача"]),
    ("{d} стояла за забором", ["дача"]),
    ("{d} молчала", ["дача"]),
    ("{d} выглядела разрушенной усадьбой", ["дача"]),
    ("{d} стояла, утопая в сирени", ["дача"]),
    ("{d} нуждалась в ремонте", ["дача"]),
    ("Обладание {d} в те времена ценилось", ["дачей"]),
    ("Он дорожил своей {d}", ["дачей"]),
    ("Она гордилась новой {d}", ["дачей"]),
    ("{d} в генеральском посёлке", ["дача", "дачи", "даче"]),
    ("{d} под Москвой", ["дача", "дачи"]),
    ("{d} в Люберцах пришлось оставить", ["дача", "дачу"]),
    ("{d} в Лесном", ["дача", "дачи"]),
]

# «Дача + одушевлённое в родительном» — посессорная конструкция
# (дача министра, на даче отца). Не путать с объектным родительным
# (дача показаний, дача взятки).
house_possessor_names = [
    "министра", "генерала", "директора", "президента",
    "художника", "писателя", "прокурора", "соседа",
    "брата", "отца", "деда", "матери", "сестры",
    "академика", "профессора", "полковника", "чиновника",
    "помещика", "купца", "графа", "князя", "хозяина",
    "знакомого", "друга", "тёти", "дяди", "бабушки", "дедушки",
    "Крэнстона", "Сталина", "родителей",
]
house_possessors = []
for name in house_possessor_names:
    house_possessors.append(f"дача {name}")
    house_possessors.append(f"на даче {name}")

# Формальный и литературный регистр: архивные описи, канцелярия, проза.
# Основной корпус НКРЯ — литературная проза, поэтому такие примеры важны.
house_formal_register = [
    "В описи имущества, составленной после смерти помещика, значились господский дом, две дачи и обширный парк с прудами",
    "Согласно архивным документам, несколько казённых дач были распределены между чиновниками ещё в довоенные годы",
    "Ведомственные дачи, выстроенные в шестидесятые годы, постепенно перешли в частную собственность их обитателей",
    "Бывшие барские дачи, переоборудованные под коммунальные квартиры, ветшали год от года и приходили в запустение",
    "Дача эта, принадлежавшая когда-то известному профессору, сохранила до наших дней первоначальную планировку фасада",
    "Наследники долго судились за право владения дачей, но в итоге продали её с торгов за долги покойного",
    "Особняк с прилегающими дачами был выставлен на торги после банкротства владельца, и желающих нашлось немного",
    "Администрация посёлка постановила снести несколько аварийных дач, признанных непригодными для проживания",
    "Сделка купли-продажи дачи была оформлена нотариально, однако регистрация затянулась почти на полгода",
    "В материалах дела упоминается дача, приобретённая обвиняемым на средства, полученные преступным путём",
    "Деревянная дача, выстроенная ещё до революции, сохранила резные наличники и высокое крыльцо с колоннами",
    "Дача стояла на отшибе, окружённая вековыми соснами, и казалась скорее лесным скитом, чем загородным домом",
    "Осенний свет ложился на пустые дачи, на покосившиеся заборы и облетевшие сады, и во всём этом было что-то щемящее",
    "За рекой, на другом берегу, белели дачи, и оттуда доносились звуки рояля и женский смех",
    "Старая дача с мезонином, где прошло его детство, теперь принадлежала чужим людям, и он не решался туда зайти",
    "Дачи тянулись вдоль дороги одна за другой, и в сумерках их силуэты напоминали сказочные терема",
    "По вечерам на дачах зажигались огни, и было слышно, как где-то далеко играют на гитаре",
    "Он вспоминал лето, проведённое на даче, и запах сосен, и голос матери, читавшей ему вслух",
    "Дача была построена по проекту известного архитектора, и в её облике угадывались черты северного модерна",
    "Обладание дачей всегда было для русской интеллигенции чем-то большим, чем просто имуществом",
    "Дача представлялась ему островком спасения от городской суеты, местом, где можно наконец подумать о главном",
    "В её представлении дача была не просто домом, а целым миром, населённым призраками прошлого",
    "Он думал о том, что дачи, как и люди, стареют, дряхлеют и уходят из жизни, оставляя после себя лишь воспоминания",
    "Решение о выделении земельных участков под дачи принималось на уровне районной администрации",
    "Государственные дачи, закреплённые за научными учреждениями, предоставлялись сотрудникам на время отпуска",
    "Порядок предоставления ведомственных дач был строго регламентирован и зависел от занимаемой должности",
    "Многие дачи, построенные в советские годы, сегодня числятся в реестре объектов культурного наследия",
    "Комиссия осмотрела дачи и признала их состояние удовлетворительным, несмотря на явные следы времени",
]

# Живые повествовательные HOUSE с бытовой «дачей».
# Сюда же — длинные HOUSE-примеры с оборотом «лесная дача».
house_long_domestic = [
    "В прошлые выходные мы всей семьёй поехали на дачу и провели там весь день",
    "Он приезжает на дачу только летом, когда в городе становится слишком душно",
    "Перевез на дачу книги и мебель, чтобы оборудовать там кабинет",
    "Приглашал на свою дачу за город коллег и старых друзей",
    "Они снимали дачу на всё лето, чтобы дети могли дышать свежим воздухом",
    "Родители решили провести лето на даче, подальше от городского шума",
    "Дед жил на даче круглый год, даже зимой не переезжал в город",
    "На даче всегда много работы: то забор поправить, то крышу перекрыть",
    "На даче растут яблони, смородина и несколько кустов крыжовника",
    "Он оборудовал мастерскую на даче и по вечерам что-то строгал",
    "Вернулись с дачи поздно вечером, еле успели на последнюю электричку",
    "Привезли с дачи варенье, яблоки и банку солёных огурцов",
    "Дорога с дачи занимает два часа, если не попасть в пробку",
    "Тропинка вела к даче через лес и небольшое поле",
    "К даче пристроили веранду, и теперь там можно пить чай в дождь",
    "Стояли у дачи и разговаривали, пока не стемнело",
    "Проезжали мимо дачи два раза, но так и не решились зайти",
    "Остановились около дачи, чтобы спросить дорогу",
    "Дача соседа была за забором, и мы почти не виделись",
    "Владелец дачи приехал на выходные и сразу взялся за дела",
    "Хозяин дачи продал участок и переехал в другой город",
    "Сосед по даче оказался врачом и часто помогал нам с детьми",
    "Старая дача покосилась от времени, но её всё ещё ремонтировали",
    "Новая дача стояла у самого леса, и по утрам там пели птицы",
    "Деревянная дача сгорела зимой, остался только фундамент",
    "Заброшенная дача заросла бурьяном и крапивой",
    "Обладание дачей в те времена считалось признаком благополучия",
    "Он дорожил своей дачей больше, чем городской квартирой",
    "Она гордилась новой дачей и часто приглашала туда гостей",
    "Государственные дачи сохраняли за семьями даже после отставки",
    "Ведомственные дачи распределяли по решению руководства",
    "Казённые дачи числились на балансе института",
    "Академические дачи давали профессуре и академикам",
    "Большим чинам такие дачи давали, а нам и не снилось",
    "Призывники строили генералам дачи, а сами жили в казармах",
    "Землю раздали под дачи, и через год там уже стояли дома",
    "План лесной дачи начертили ещё в прошлом году, но строить так и не начали",
    "Продажа лесной дачи затянулась из-за бумажных проволочек, но в итоге всё устроилось",
    "Лесная дача стояла на краю бора, и по утрам там пахло соснами и грибами",
    "Хозяин лесной дачи часто приезжал туда один, без семьи, и подолгу гулял по лесу",
    "Лесную дачу окружали вековые ели, и даже в солнечный день там было сумрачно",
    "В лесной даче было всего три комнаты, но зато какая вокруг тишина",
    "Он вспоминал лесную дачу из детства: скрип половиц, запах дыма и долгие прогулки",
    "Лесная дача досталась ей от бабушки, и она ни за что не хотела её продавать",
]

# Формы множественного числа («дачами», «дачах», «дач») в перечислениях
# и перефразах. В литературной прозе такие конструкции встречаются часто.
house_long_plural = [
    "Люди казались занятыми только заработками, дачами, машинами и прочими благами",
    "Он мечтал не о курортах и дачах, а о простом человеческом отдыхе",
    "На стенах дач и на заборах висели яркие афиши предстоящих концертов",
    "Раздавали велосипеды и дачи всем желающим — такова была программа праздника",
    "Обитатели дач выходили по утрам на прогулку к реке и здоровались с соседями",
    "Среди старых дач и новых коттеджей легко было заблудиться",
    "По крышам дач и сараев стучал осенний дождь, и в окнах зажигался свет",
    "Вокруг дач и посёлков тянулись бесконечные поля и перелески",
    "За заборами дач лаяли собаки, и слышались голоса детей",
    "Дачи в этом месте были старые, но ухоженные, с крепкими заборами",
    "Мимо дач и посёлков проносились редкие электрички и товарные поезда",
    "Жизнь на дачах и в окрестных деревнях текла размеренно и тихо",
    "На дачах и в соседних деревнях уже начали собирать урожай",
    "Владельцы дач и коттеджей съезжались к выходным и заполняли электрички",
    "С дач и из соседних посёлков люди каждое утро ехали в город на работу",
]

# «Дачу + глагол действия» (отберут, продадут, построили, купили,
# сожгли) — HOUSE. Отличать от формул с дачей-действием: «дачу розог»,
# «дачу взятки».
house_actions = [
    "Дачу отберут", "Дачу отберут по закону", "Дачу отберут за долги",
    "Дачу продадут", "Дачу продадут и переедут в город",
    "Дачу снесут", "Дачу снесут из-за стройки",
    "Дачу оставят", "Дачу оставят наследникам",
    "Дачу построили", "Дачу построили за одно лето",
    "Дачу сожгли", "Дачу сожгли и не восстановили",
    "Дачу купили", "Дачу купили у знакомых",
    "Дачу сдадут", "Дачу сдадут на лето",
    "Эту дачу не узнать", "Дачу отстояли",
    "Дачу перестроили", "Дачу завещали детям",
    "Дачу облагородили", "Дачу арендовали",
    "Дачу утеплили", "Дачу застраховали",
]

# Длинные повествовательные HOUSE с глаголами движения и пребывания
# (гнать, уехать, возвращаться, приглашать). Такие фразы часто
# встречаются в литературной прозе и легко смешиваются с OTHER в
# длинном контексте.
house_long_context = [
    "Она уехала на дачу на всё лето, оставив квартиру на попечение соседей",
    "Они уехали на дачу в пятницу вечером и вернулись только в воскресенье",
    "Вся семья уехала на дачу сразу после последнего звонка, и город опустел",
    "Уехав на дачу, он редко звонил и почти не писал писем",
    "Отец гнал его на дачу каждое лето, чтобы он там дышал свежим воздухом",
    "Мать гнала нас на дачу, как только заканчивалась школа",
    "Жена гнала его на дачу по субботам, хотя ему хотелось остаться в городе",
    "Гнали его на дачу и родные, и знакомые, но он упирался",
    "На даче летом они устраивали музыкальные вечера, и старый рояль снова звучал",
    "В гостиной на даче стоял рояль, на котором играла ещё бабушка",
    "По вечерам на даче пели романсы под гитару, и голоса разносились над рекой",
    "На даче, летним вечером, они слушали, как играет рояль, и не хотели уходить",
    "Он приглашал для серьёзных разговоров на свою дачу только самых близких",
    "Приглашал на дачу для серьёзных разговоров, но не для праздников",
    "Она приглашала для серьёзных разговоров на свою дачу, где никто не мешал",
    "Для серьёзных разговоров он всегда приглашал на дачу, подальше от суеты",
    "Мать гнала его с дачи в город, но он упирался и оставался до первых заморозков",
    "Хозяин гнал усталую труппу с дачи, не давая артистам даже собрать вещи",
    "Они выходили с дачи поздно вечером и шли к станции через тёмный лес",
    "Выходя с дачи, она всякий раз оборачивалась, словно прощаясь с кем-то",
    "Он возвращался с дачи усталый, но довольный, и сразу засыпал в кресле",
    "Возвращались с дачи по пустынному шоссе, и фары выхватывали из темноты кусты",
    "Она неожиданно уехала на дачу, никому ничего не сказав и не попрощавшись",
    "На этой даче всем было легко и непринуждённо, и никто не хотел уезжать",
    "На этой даче, среди старых сосен и яблонь, дышалось особенно свободно",
    "На даче, где стоял старый рояль, часто собирались гости и музицировали",
    "Брат гнал его на дачу, чтобы тот помог с ремонтом, но он отнекивался",
]

# Адресные короткие HOUSE-фразы: закрывают тематические случаи, где
# «дача» легко смешивается с OTHER.
house_short_targeted = [
    # Короткие назывные заголовки
    "Старая дача",
    "Заброшенная дача",
    "Дача у озера",
    "Дача у реки",
    "Дача в лесу",
    "Дача на продажу",
    "Дача продаётся",
    "Дача в аренду",
    "Наша дача",
    "Дача зимой",
    "Дача осенью",
    "Загородная дача",
    "Дача в Подмосковье",
    # Метаязыковые определения
    "Дачи — это загородные дома",
    "Дача — это дом за городом",
    "Слово «дача» означает загородный дом",
    "В словаре «дача» переводится как загородный дом",
    # «Объяснение» рядом с HOUSE, а не «дача объяснений»
    "После объяснения они уехали на дачу",
    "Объяснение произошло на даче, а не в городе",
    "Он ждал объяснения на даче, но так и не дождался",
    "На даче, после объяснения, всё стало ясно",
    # «раздача/раздали» + дач* — «дача» здесь существительное (дома)
    "Раздача дач и участков началась в понедельник",
    "Раздали дачи и путёвки всем желающим",
    "Бесплатная раздача дач и велосипедов",
    "Раздача дач шла целый день",
    # «Дача в генеральском посёлке»
    "Дача в генеральском посёлке стояла на холме",
    "Рядом с дачей в генеральском посёлке был пруд",
]


# ---------------------------------------------------------------------------
# 3. Train: OTHER
# ---------------------------------------------------------------------------
other_pairs = [
    ("Свидетеля вызвали для {d} показаний", ["дачи"]),
    ("Суд обязал его явиться для {d} показаний", ["дачи"]),
    ("Без {d} показаний дело не рассмотреть", ["дачи"]),
    ("В процессе {d} показаний он путался", ["дачи"]),
    ("После {d} показаний его отпустили", ["дачи"]),
    ("Ему вменяют {d} ложных показаний", ["дачу"]),
    ("За {d} ложных показаний дали срок", ["дачу"]),
    ("Обвиняется в {d} заведомо ложных показаний", ["даче"]),
    ("Его обвинили в {d} ложных показаний", ["даче"]),
    ("Факт {d} заведомо ложных показаний", ["дачи"]),
    ("пройти через процедуру {d} показаний", ["дачу"]),
    ("Слушание отложили до {d} показаний", ["дачи"]),
    ("За {d} взятки ему дали срок", ["дачу"]),
    ("Факт {d} взятки доказан", ["дачи"]),
    ("Его задержали при {d} взятки", ["даче"]),
    ("признался в {d} взятки", ["даче"]),
    ("Уличен в {d} взятки", ["даче"]),
    ("Скотине устроили {d} корма", ["дачу"]),
    ("Провинившемуся назначили {d} розог", ["дачу"]),
    ("Его судили за {d} яду", ["дачу"]),
    ("Объявление о {d} обедов", ["даче"]),
    ("Банк подтвердил {d} ссуды под залог", ["дачу"]),
    ("Дача отсрочки платежа оформлена актом", ["дача"]),
    ("Матерщина, угрозы, {d} «дел»", ["дача"]),
    ("Кузьма чуял большую {d} на водку", ["дачу"]),
    ("Дать ему {d} на чай", ["дачу"]),
]

other_formulas = [
    "дача розог", "дача яду", "дача пощёчины", "дача пинка",
    "дача оплеухи", "дача тумаков", "дача сдачи",
    "дача клятвы", "дача обета", "дача сигнала",
    "дача разрешения", "дача обещания", "дача поручения",
    "дача показаний", "дача взятки", "дача корма",
    "дача обедов", "дача ссуды", "дача отсрочки",
    "дача ложных показаний", "дача заведомо ложных показаний",
    "дача крупной взятки", "дача денег", "дача денежных средств",
    "дача денежного вознаграждения",
    "дачу розог", "дачу яду", "дачу пощёчины", "дачу пинка",
    "дачу оплеухи", "дачу тумаков", "дачу сдачи",
    "дачу клятвы", "дачу обета", "дачу сигнала",
    "дачу разрешения", "дачу обещания", "дачу поручения",
    "дачу на водку", "дачу на чай", "дачу на прощание",
    "выдали дачу розог", "получил дачу розог",
    "назначили дачу розог", "дал дачу яду",
    "дал дачу пощёчины", "дал дачу пинка",
    "дача сухого фуража", "дача порции хлеба", "дача сведений",
    "дача объяснений", "дача интервью", "дача аванса",
    "дача гарантии", "дача займа", "дача кредита",
    "дача приказа", "дача согласия", "дача клятвы верности",
    "дача фуража", "дачу фуража", "даче фуража",
    "дачей фуража", "дача овса", "дачу овса",
    "дача домашних обедов", "дача вкусных обедов",
    "дача горячих обедов", "дача сытных обедов",
    "дачу домашних обедов", "дачу вкусных обедов",
]

other_long_canonical = [
    "Свидетеля вызвали для дачи показаний, но он так и не явился",
    "Суд обязал его явиться для дачи показаний в назначенный срок",
    "Без дачи показаний дело не рассмотреть, поэтому заседание отложили",
    "В процессе дачи показаний он путался и противоречил сам себе",
    "После дачи показаний его отпустили под подписку о невыезде",
    "Ему вменяют дачу ложных показаний и воспрепятствование следствию",
    "За дачу ложных показаний ему дали реальный срок",
    "Обвиняется в даче заведомо ложных показаний в суде",
    "Его обвинили в даче ложных показаний следователям",
    "Факт дачи заведомо ложных показаний был доказан в суде",
    "Теперь вы готовы пройти через процедуру дачи показаний",
    "Слушание отложили до дачи показаний ключевого свидетеля",
    "За дачу взятки должностному лицу ему дали срок",
    "Факт дачи взятки доказан, осталось установить сумму",
    "Его задержали при даче взятки сотруднику полиции",
    "Он признался в даче взятки и сотрудничает со следствием",
    "Уличен в даче взятки крупному чиновнику",
    "Скотине устроили дачу корма, но она всё равно голодала",
    "Провинившемуся назначили дачу розог при свидетелях",
    "Его судили за дачу яду Смелькову с умыслом на убийство",
    "Объявление о даче обедов висело на дверях столовой",
    "Банк подтвердил дачу ссуды под залог недвижимости",
    "Дача отсрочки платежа оформлена актом и подписана сторонами",
    "Матерщина, угрозы, дача «дел» — всё это годилось для заключённых",
    "Кузьма уже чуял большую дачу на водку и суетился",
    "Он любил дать кому-нибудь дачу на чай, чтобы выглядеть щедрым",
    "В процессе дачи показаний выяснились новые обстоятельства",
    "Его подозревают в даче ложных показаний под присягой",
    "Его признали виновным в даче ложных показаний на предварительном следствии",
    "Обвиняемый признал себя виновным в даче заведомо ложных показаний",
    "Свидетели были вызваны для дачи показаний в суд по делу о растрате",
    "Следователь указал место дачи показаний в протоколе допроса",
    "Защита ходатайствовала о даче свидетельских показаний в закрытом режиме",
    "Его осудили за дачу ложных показаний ФБР и воспрепятствование правосудию",
    "В постановлении суда упоминается дача показаний свидетелем обвинения",
]

other_long_corporal = [
    "Провинившемуся дали с излишком пересыпанную дачу розог",
    "Приговор гласил: дачу розог и высылку в деревню на поселение",
    "За дерзость ему прописали дачу розог при всех, и он молча стерпел",
    "Солдату дали дачу розог, но он даже не вскрикнул",
    "Его обвиняли в даче яду сопернику, но доказать ничего не смогли",
    "Подсудимый отрицал дачу яду, хотя улики говорили об обратном",
    "Доктор подозревал дачу яду, но следов отравы не нашли",
    "За дачу розог на корабле наказывали строже, чем за кражу",
    "Дачу розог ему отвесили с излишком, и он запомнил это надолго",
]

other_long_provisions = [
    "В столовой висело объявление о даче обедов по сниженным ценам для рабочих",
    "Рабочим обещали дачу горячих обедов прямо в новом цехе",
    "Благотворительное общество занималось дачей бесплатных обедов беднейшим жителям",
    "Кони, получившие свою дачу овса, мирно стояли в стойлах и жевали сено",
    "Скотине выдавали дачу сена и корнеплодов два раза в день",
    "Лошади, получив дачу овса, успокоились и больше не били копытами",
    "Лошадям выдали положенную дачу сухого фуража, и они успокоились",
    "Смотритель отмерил лошадям дачу сухого фуража и ушёл спать",
]


# ---------------------------------------------------------------------------
# 4. Validation
# ---------------------------------------------------------------------------
valid_house = [
    "Поехали на дачу в субботу утром",
    "Дом отдыха стоял рядом с дачей Крэнстонов",
    "Он всю жизнь мечтал о собственной даче",
    "Дача сгорела, но её восстановили",
    "На даче под Москвой собрались друзья",
    "Привезли с дачи мешок картошки",
    "К даче вела узкая тропинка",
    "Остановились у дачи, чтобы передохнуть",
    "Обладание дачей считалось привилегией",
    "Государственные дачи распределяли по разнарядке",
    "Он продал дачу и купил квартиру в городе",
    "Старая дача стояла у самого леса",
    "Сосед по даче оказался бывшим военным",
    "Дача в Лесном пустовала всю зиму",
    "Дача номер три занимал врач",
    "Призывники строили генералам дачи",
    "Ведомственные дачи сохраняли за семьями",
    "Дача Сталина в Сочи теперь музей",
    "Они снимали дачу на всё лето",
    "На даче росли старые яблони",
    "Лесная дача стояла на краю бора, и оттуда был виден весь посёлок",
    "Лесную дачу сдавали на лето семьям с детьми",
    "Летом вся семья перебирается на дачу, где дед целыми днями возится с яблонями, а бабушка варит варенье",
    "Дорога до дачи занимала два часа, но мы всё равно ездили каждые выходные, потому что дети обожали там бегать",
    "Когда-то эта дача принадлежала профессору, теперь её сдавали на лето студентам и молодым семьям",
    "Мама настаивала, чтобы мы провели выходные на даче, а не в душной городской квартире, и мы в итоге согласились",
    "На старой даче до сих пор сохранилась веранда, где прадед любил сидеть с газетой и слушать птиц",
    "Зимой дача стояла пустая, но каждую весну отец приезжал первым, чтобы проверить крышу и затопить печь",
    "Мы долго выбирали, какую дачу купить: хотелось и близко к реке, и не слишком далеко от города",
    "В детстве я проводил каждое лето на даче у бабушки, и запах сосен там до сих пор для меня — запах каникул",
    "Хозяин дачи предупреждал, что вода из колодца не для питья, но мы всё равно её кипятили и пили чай",
    "Новая дача стояла почти у самого леса, и по утрам на подоконник слетались синицы, требуя крошек",
    "Соседская дача давно была заброшена, и мы с братом лазали через забор, чтобы посмотреть, что внутри",
    "Дачу пришлось продать после переезда, и до сих пор родители вспоминают её как лучшее место на свете",
    "Каждый август на даче начинался марафон: закатки, сушка яблок, уборка урожая и вечные споры о том, что сажать в следующем году",
    "Вечером на даче пахло дымом от соседского самовара, и казалось, что лето никогда не кончится",
    "Дача эта была неказистая, но крепкая: сруб из лиственницы, печь, сложенная ещё прадедом, и скрипучие половицы",
    "Дача эта, оставленная прежними хозяевами, поражала запустением: окна заколочены, крыша провалилась, а сад одичал",
    "Ведомственная дача, выделенная отцу от института, стояла в живописном месте, и туда каждое лето съезжались коллеги",
    "В газетной заметке сообщалось, что несколько дач в посёлке будут снесены из-за расширения дороги",
    "Повесть заканчивалась сценой прощания с дачей, куда герой больше никогда не вернётся",
    "Уехав на дачу в начале июня, они вернулись только к сентябрю",
    "Она уехала на дачу, не попрощавшись, и это всех обидело",
    "Отец гнал нас на дачу с раннего утра, чтобы успеть до жары",
    "Его гнали на дачу и работа, и семья, и собственное желание отдохнуть",
    "За заборами дач лаяли собаки и слышались голоса",
    "На дачах и в окрестных сёлах было тихо и пусто",
    "Дачами здесь никого не удивишь: они стояли чуть ли не в каждом дворе",
    "По вечерам на дачах зажигали свет, и было видно, как в окнах двигаются силуэты",
    "Среди дач и огородов петляла узкая тропинка к реке",
]

valid_other = [
    "Свидетеля вызвали для дачи показаний в суд",
    "Его обвинили в даче ложных показаний",
    "За дачу взятки должностному лицу дали срок",
    "Факт дачи взятки установлен следствием",
    "Скотине устроили дачу корма",
    "Провинившемуся назначили дачу розог",
    "Его судили за дачу яду",
    "Банк подтвердил дачу ссуды",
    "Объявление о даче обедов",
    "Речь шла о даче обещания помочь",
    "Дача отсрочки платежа оформлена актом",
    "Матерщина, угрозы, дача «дел»",
    "Кузьма чуял большую дачу на водку",
    "В процессе дачи показаний он путался",
    "Слушание отложили до дачи показаний",
    "От дачи каких-либо показаний он отказался",
    "Свидетеля привлекли к ответственности за дачу ложных показаний",
    "Следствие установило факт дачи взятки",
    "Подсудимый отрицал дачу взятки",
    "За дачу заведомо ложных показаний предусмотрена статья",
    "Речь шла о даче обещаний, которые никто не собирался выполнять",
    "Наказание за дачу взятки ужесточили",
    "Процедура дачи показаний заняла несколько часов",
    "Обвинение основывалось на даче показаний двумя свидетелями",
    "Дача корма скоту входила в его ежедневные обязанности",
    "Задержанный признался в даче взятки инспектору",
    "Суд рассматривал дело о даче ложных показаний",
    "Следователь допрашивал свидетеля перед дачей показаний",
    "Документы подтверждали дачу разрешения на строительство",
    "Никто не помнил о даче обещания вернуться",
    "В романе упоминалась дача пощёчины обидчику",
    "Получившие свою дачу сухого фуража лошади мирно жевали в стойлах",
    "Кони получили свою дачу овса и успокоились",
]


# ---------------------------------------------------------------------------
# 5. Сборка и фильтрация утечек
# ---------------------------------------------------------------------------
def expand(pairs):
    """Разворачивает список (шаблон, [формы]) в плоский список строк."""
    out = []
    for tpl, forms in pairs:
        for f in forms:
            out.append(tpl.format(d=f))
    return out


def dedup_internal(items):
    """Удаляет точные дубликаты по нормализованному ключу."""
    seen, out = set(), []
    for t in items:
        k = norm_hard(t)
        if k in seen:
            continue
        seen.add(k)
        out.append(t)
    return out


def drop_test_leaks(items, name):
    """
    Убирает train/valid-примеры, которые могут дать утечку в тест:
      - точное совпадение по нормализованному тексту
      - Jaccard ≥ 0.7 с любой тестовой строкой
      - train-фраза (≥5 слов) целиком входит в тестовую (substring)
    """
    kept = []
    dropped = {"exact": [], "jaccard": [], "substring": []}
    for t in items:
        s_hard = norm_hard(t)
        s_soft = norm_soft(t)
        if s_hard in test_hard_set:
            dropped["exact"].append(t)
            continue
        A = nset(t)
        if any(jac_set(A, B) >= 0.7 for B in test_sets):
            dropped["jaccard"].append(t)
            continue
        if len(s_soft.split()) >= 5 and any(s_soft in x for x in test_soft_list):
            dropped["substring"].append(t)
            continue
        kept.append(t)
    print(f"\n  {name}: {len(items)} → {len(kept)}  "
          f"(exact={len(dropped['exact'])}, "
          f"jaccard={len(dropped['jaccard'])}, "
          f"substring={len(dropped['substring'])})")
    for k in dropped:
        for t in dropped[k][:2]:
            print(f"    [{k}] {t[:100]}")
    return kept


def jac_dedup_vs(items, refs, thr=0.7, name=""):
    """Убирает из valid-примеров те, что Jaccard ≥ thr близки к train."""
    ref_sets = [nset(r) for r in refs]
    kept, dropped = [], 0
    for t in items:
        A = nset(t)
        if any(jac_set(A, B) >= thr for B in ref_sets):
            dropped += 1
            continue
        kept.append(t)
    print(f"  {name}: удалено {dropped}, осталось {len(kept)}")
    return kept


print("\n" + "=" * 60)
print("СБОРКА TRAIN")

house_raw = (
    expand(house_pairs)
    + house_possessors
    + house_formal_register
    + house_long_domestic
    + house_long_plural
    + house_actions
    + house_long_context
    + house_short_targeted
)
house_raw = dedup_internal(house_raw)
print(f"  HOUSE до фильтра: {len(house_raw)}")
house_texts = drop_test_leaks(house_raw, "HOUSE")

other_raw = (
    expand(other_pairs)
    + other_formulas
    + other_long_canonical
    + other_long_corporal
    + other_long_provisions
)
other_raw = dedup_internal(other_raw)
print(f"\n  OTHER до фильтра: {len(other_raw)}")
other_texts = drop_test_leaks(other_raw, "OTHER")

print("\n" + "=" * 60)
print("СБОРКА VALID")
valid_h = drop_test_leaks(dedup_internal(valid_house), "VALID_HOUSE")
valid_o = drop_test_leaks(dedup_internal(valid_other), "VALID_OTHER")

train_pool = house_texts + other_texts
valid_h = jac_dedup_vs(valid_h, train_pool, name="VALID_HOUSE vs train")
valid_o = jac_dedup_vs(valid_o, train_pool, name="VALID_OTHER vs train")


# ---------------------------------------------------------------------------
# 6. Финальный контроль
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("КОНТРОЛЬ")

h_keys = set(norm_hard(t) for t in house_texts)
o_keys = set(norm_hard(t) for t in other_texts)
cross = h_keys & o_keys
print(f"  HOUSE ∩ OTHER: {len(cross)}")

train_hard_set = set(norm_hard(t) for t in train_pool)
train_sets = [nset(t) for t in train_pool]
n_exact = sum(1 for t in test_df["text"] if norm_hard(t) in train_hard_set)
n_jacc = sum(1 for t in test_sets
             if any(jac_set(t, trs) >= 0.7 for trs in train_sets))
print(f"  exact train↔test:    {n_exact}")
print(f"  Jaccard train↔test:  {n_jacc}")

n_short_h = sum(1 for t in house_texts if len(t) < 60)
n_short_o = sum(1 for t in other_texts if len(t) < 60)
print(f"\n  TRAIN HOUSE: коротких {n_short_h}, длинных {len(house_texts) - n_short_h}")
print(f"  TRAIN OTHER: коротких {n_short_o}, длинных {len(other_texts) - n_short_o}")


# ---------------------------------------------------------------------------
# 7. Сохранение
# ---------------------------------------------------------------------------
# Формат меток по файлам (проектная конвенция, см. README):
#   train_house.csv, train_other.csv, valid.csv — label: 0/1
#   test.csv — reference: HOUSE/OTHER
print("\n" + "=" * 60)
print("СОХРАНЕНИЕ")

pd.DataFrame({"text": house_texts, "label": 0}).to_csv(
    DATA_DIR / "train_house.csv", sep=";", encoding="utf-8-sig", index=False)
pd.DataFrame({"text": other_texts, "label": 1}).to_csv(
    DATA_DIR / "train_other.csv", sep=";", encoding="utf-8-sig", index=False)

valid_df = pd.DataFrame({
    "text": valid_h + valid_o,
    "label": [0] * len(valid_h) + [1] * len(valid_o),
})
valid_df.to_csv(DATA_DIR / "valid.csv", sep=";", encoding="utf-8-sig", index=False)

test_df.to_csv(DATA_DIR / "test.csv", sep=";", encoding="utf-8-sig", index=False)

print(f"  train_house.csv: {len(house_texts)} строк  (label 0)")
print(f"  train_other.csv: {len(other_texts)} строк  (label 1)")
print(f"  valid.csv:       {len(valid_df)} (H={len(valid_h)}, O={len(valid_o)})")
print(f"  test.csv:        {len(test_df)} "
      f"(H={(test_df.reference=='HOUSE').sum()}, "
      f"O={(test_df.reference=='OTHER').sum()})  [reference]")


# ---------------------------------------------------------------------------
# Итог
# ---------------------------------------------------------------------------
ok = (n_exact == 0 and n_jacc == 0 and len(cross) == 0)
print(f"\n  {'✓ Готово к обучению' if ok else '✗ Проверьте утечки'}")

In [ ]:
# ============================================================================
# 2. Обучение SetFit + подбор порога на valid
# ============================================================================
"""
Обучает бинарный классификатор HOUSE / OTHER на синтетическом train.

Модель: SetFit поверх DeepPavlov/rubert-base-cased. SetFit дообучает
sentence-transformer на парах «позитив / негатив», затем подгоняет
логистическую регрессию на эмбеддингах. Для нескольких сотен примеров
это экономичнее, чем полный fine-tuning BERT-головы.

Гиперпараметры: num_epochs=2, num_iterations=16, batch_size=16,
max_seq_length=256, seed=42.

Порог бинаризации выбирается на valid:
  - HOUSE recall ≥ 0.90 обязателен (не хочется терять HOUSE-примеры
    из-за того, что модель перестраховалась в пользу OTHER);
  - если valid насыщен (macro_F1 = 1.0 на всём диапазоне) — берём 0.50
    a priori: это честнее, чем выбирать крайнее значение из плато;
  - иначе — порог с максимальным macro_F1 среди допустимых.

Вход:  train_house.csv, train_other.csv, valid.csv
Выход: model/setfit_dacha_classifier/ — веса и training_config.json

Три источника модели, в порядке приоритета:
  1. локальная папка model/setfit_dacha_classifier/ — если сохранена
     после предыдущего запуска;
  2. Hugging Face Hub — bursansan/dacha-setfit-rubert-base — если
     локальной нет, модель скачивается и сохраняется локально;
  3. обучение с нуля — только если SKIP_IF_EXISTS = False.

Переключатель SKIP_IF_EXISTS:
  True  — использовать готовую модель (локально или с HF);
  False — обучить заново (перезапишет сохранённую модель).
"""

import json
import random
from pathlib import Path

# setfit может отсутствовать при локальном запуске без ячейки 0.
try:
    import setfit  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "setfit", "transformers", "datasets", "scikit-learn"])

import numpy as np
import pandas as pd
from datasets import Dataset
from huggingface_hub import snapshot_download
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.metrics import f1_score, recall_score

# --- 0. Расположение данных и модели ---
# DATA_DIR определяется в ячейке 0 (Setup). Если ячейка запускается
# изолированно — находим data/ автоматически, поднимаясь вверх от CWD.
if "DATA_DIR" not in globals():
    for _p in (Path.cwd(), *Path.cwd().parents):
        if (_p / "data" / "raw" / "ruscorpora_content.csv").exists():
            DATA_DIR = _p / "data"
            break
    else:
        raise FileNotFoundError(
            "Не найдена папка data/. Запустите ячейку 0 (Setup) "
            "или откройте ноутбук из корня репозитория."
        )

REPO_ROOT   = DATA_DIR.resolve().parent
MODEL_DIR   = REPO_ROOT / "model" / "setfit_dacha_classifier"
CONFIG_FILE = MODEL_DIR / "training_config.json"

# Публичный репозиторий модели на Hugging Face Hub. Используется, если
# локальной папки нет. После первого скачивания модель оказывается в
# MODEL_DIR, и следующий запуск идёт по локальной ветке.
HF_REPO = "bursansan/dacha-setfit-rubert-base"

print(f"DATA_DIR  = {DATA_DIR}")
print(f"MODEL_DIR = {MODEL_DIR}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MODEL_NAME     = "DeepPavlov/rubert-base-cased"
MAX_SEQ_LENGTH = 256
NUM_EPOCHS     = 2
NUM_ITERATIONS = 16
BATCH_SIZE     = 16

SKIP_IF_EXISTS = True   # False → переобучить даже если модель уже есть


def model_is_saved():
    """Модель считается сохранённой, если есть веса, head и конфиг порога."""
    weights = ((MODEL_DIR / "model.safetensors").exists()
               or (MODEL_DIR / "pytorch_model.bin").exists())
    head    = (MODEL_DIR / "model_head.pkl").exists()
    return weights and head and CONFIG_FILE.exists()


# --- 1. Загрузка данных ---
print("\n" + "=" * 60)
print("1. ЗАГРУЗКА")

df_h = pd.read_csv(DATA_DIR / "train_house.csv", sep=";", encoding="utf-8-sig")
df_o = pd.read_csv(DATA_DIR / "train_other.csv", sep=";", encoding="utf-8-sig")
train_df = (pd.concat([df_h, df_o], ignore_index=True)
              .sample(frac=1, random_state=SEED)
              .reset_index(drop=True))
valid_df = pd.read_csv(DATA_DIR / "valid.csv", sep=";", encoding="utf-8-sig")

n_h = int((train_df.label == 0).sum())
n_o = int((train_df.label == 1).sum())
v_h = int((valid_df.label == 0).sum())
v_o = int((valid_df.label == 1).sum())

print(f"  Train: {len(train_df)}  (H={n_h}, O={n_o})")
print(f"  Valid: {len(valid_df)}  (H={v_h}, O={v_o})")
print(f"  Model: {MODEL_NAME}")

# Быстрая проверка, что train и valid не пересекаются по тексту.
def nh(s):
    return " ".join(str(s).split()).lower()

tr = set(train_df["text"].map(nh))
va = set(valid_df["text"].map(nh))
if tr & va:
    raise SystemExit(f"train ∩ valid: {len(tr & va)} — обучение остановлено")
print("  train ∩ valid: 0 ✓")


# --- 2. Модель: локально / с HF / обучение ---
if SKIP_IF_EXISTS and model_is_saved():
    print("\n" + "=" * 60)
    print("2. ЗАГРУЗКА ГОТОВОЙ МОДЕЛИ (локально)")
    model = SetFitModel.from_pretrained(str(MODEL_DIR))
    with open(CONFIG_FILE, encoding="utf-8") as f:
        training_config = json.load(f)
    best_thr = training_config["threshold"]
    thr_note = training_config.get("threshold_note", "loaded from disk")
    print(f"  Модель: {MODEL_DIR}")
    print(f"  Порог:  {best_thr}  ({thr_note})")

elif SKIP_IF_EXISTS:
    print("\n" + "=" * 60)
    print("2. ЗАГРУЗКА ГОТОВОЙ МОДЕЛИ (Hugging Face Hub)")
    print(f"  Локальной модели нет — скачиваю: {HF_REPO}")
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=HF_REPO,
        local_dir=str(MODEL_DIR),
    )
    model = SetFitModel.from_pretrained(str(MODEL_DIR))
    with open(CONFIG_FILE, encoding="utf-8") as f:
        training_config = json.load(f)
    best_thr = training_config["threshold"]
    thr_note = training_config.get("threshold_note", "downloaded from HF")
    print(f"  Сохранена локально: {MODEL_DIR}")
    print(f"  Порог:  {best_thr}  ({thr_note})")

else:
    print("\n" + "=" * 60)
    print("2. ОБУЧЕНИЕ")

    # Время обучения: T4 GPU ~2 минуты, CPU ~4–5 часов.
    import torch
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("  ⚠ CUDA недоступна — обучение пойдёт на CPU (~4–5 часов).")
        print("    Colab: Runtime → Change runtime type → T4 GPU.")

    train_ds = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
    valid_ds = Dataset.from_pandas(valid_df[["text", "label"]].reset_index(drop=True))

    model = SetFitModel.from_pretrained(
        MODEL_NAME,
        labels=[0, 1],
        max_seq_length=MAX_SEQ_LENGTH,
    )
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            batch_size=BATCH_SIZE,
            num_epochs=NUM_EPOCHS,
            num_iterations=NUM_ITERATIONS,
            seed=SEED,
            report_to="none",
        ),
        train_dataset=train_ds,
        eval_dataset=valid_ds,
    )
    trainer.train()

    # --- 3. Порог на valid ---
    print("\n" + "=" * 60)
    print("3. ВЫБОР ПОРОГА НА VALID")

    probs = model.predict_proba(valid_df["text"].tolist())
    if hasattr(probs, "detach"):
        probs = probs.detach().cpu().numpy()
    probs = np.asarray(probs)
    p_val = probs[:, 1] if probs.ndim == 2 else probs.ravel()
    y_val = valid_df["label"].values

    rows = []
    for thr in np.round(np.arange(0.30, 0.86, 0.05), 2):
        preds = (p_val >= thr).astype(int)
        rows.append({
            "thr":      float(thr),
            "H_recall": float(recall_score(y_val, preds, pos_label=0, zero_division=0)),
            "F1_OTHER": float(f1_score(y_val, preds, pos_label=1, zero_division=0)),
            "macro_F1": float(f1_score(y_val, preds, average="macro", zero_division=0)),
        })
    rows_df = pd.DataFrame(rows)
    print(rows_df.to_string(index=False, float_format="%.4f"))

    feasible = rows_df[rows_df["H_recall"] >= 0.90]
    if len(feasible) == 0:
        best_thr = 0.50
        thr_note = "no feasible threshold; 0.50 fallback"
    elif feasible["macro_F1"].max() >= 0.999:
        best_thr = 0.50
        thr_note = "valid saturated; 0.50 chosen a priori"
    else:
        best_row = feasible.sort_values("macro_F1", ascending=False).iloc[0]
        best_thr = float(best_row["thr"])
        thr_note = "max macro_F1 among H_recall≥0.90"

    # Метрики именно на выбранном пороге
    preds = (p_val >= best_thr).astype(int)
    metrics_at_best = {
        "H_recall": float(recall_score(y_val, preds, pos_label=0, zero_division=0)),
        "F1_OTHER": float(f1_score(y_val, preds, pos_label=1, zero_division=0)),
        "macro_F1": float(f1_score(y_val, preds, average="macro", zero_division=0)),
    }
    print(f"\n  Best thr: {best_thr}   Note: {thr_note}")
    print(f"  Valid @ best thr: macro_F1={metrics_at_best['macro_F1']:.4f}, "
          f"F1_OTHER={metrics_at_best['F1_OTHER']:.4f}, "
          f"H_recall={metrics_at_best['H_recall']:.4f}")

    # --- 4. Сохранение ---
    print("\n" + "=" * 60)
    print("4. СОХРАНЕНИЕ")

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(MODEL_DIR))

    training_config = {
        "base_model":     MODEL_NAME,
        "max_seq_length": MAX_SEQ_LENGTH,
        "num_epochs":     NUM_EPOCHS,
        "num_iterations": NUM_ITERATIONS,
        "batch_size":     BATCH_SIZE,
        "seed":           SEED,
        "threshold":      float(best_thr),
        "threshold_note": thr_note,
        "valid_metrics":  metrics_at_best,
        "train": {"size": int(len(train_df)), "H": n_h, "O": n_o},
        "valid": {"size": int(len(valid_df)), "H": v_h, "O": v_o},
        "labels": {"0": "HOUSE", "1": "OTHER"},
    }
    with open(CONFIG_FILE, "w", encoding="utf-8") as f:
        json.dump(training_config, f, ensure_ascii=False, indent=2)

    print(f"  ✓ {MODEL_DIR}/")
    print(f"  ✓ training_config.json (threshold={best_thr})")


# --- 5. Сводка (одинаковая для всех трёх веток) ---
print("\n" + "=" * 60)
print("5. СВОДКА")
print("=" * 60)
print(f"  train:     {len(train_df)} (H={n_h}, O={n_o})")
print(f"  valid:     {len(valid_df)} (H={v_h}, O={v_o})")
print(f"  threshold: {best_thr}  ({thr_note})")
print(f"  model:     {MODEL_DIR}")

print("\n  Следующий шаг: 3. Оценка модели на test.")

In [ ]:
# ============================================================================
# 3. Оценка модели на test
# ============================================================================
"""
Оценивает модель на 703 корпусных вхождениях слова «дача»
(649 HOUSE / 54 OTHER), которые не участвовали ни в обучении,
ни в подборе порога. Использует model и best_thr из ячейки 2.
На test порог не подбирается.

Дополнительно:
  - sanity-проверка отсутствия утечек train ↔ test;
  - распределение P(OTHER) и зазор между классами;
  - разбивка ошибок по авторам — для results/error_analysis.md.

Вход:  data/test.csv, data/train_house.csv, data/train_other.csv,
       data/annotated_corpus.csv.
Выход: results/test_predictions.csv
"""

import re
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
)

THRESHOLD   = float(best_thr)
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE    = RESULTS_DIR / "test_predictions.csv"

print(f"Порог:       {THRESHOLD}")
print(f"DATA_DIR:    {DATA_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")


# --- 1. Загрузка test ---
te = pd.read_csv(DATA_DIR / "test.csv", sep=";", encoding="utf-8-sig", dtype=str)

assert "reference" in te.columns, \
    "test.csv должен содержать колонку reference (HOUSE/OTHER)"
assert te["reference"].isin(["HOUSE", "OTHER"]).all(), \
    "reference должен содержать только HOUSE/OTHER"

X_test = te["text"].tolist()
y_true = te["reference"].map({"HOUSE": 0, "OTHER": 1}).to_numpy()
n_h = int((y_true == 0).sum())
n_o = int((y_true == 1).sum())
print(f"\n  test: {len(X_test)}  (H={n_h}, O={n_o})")


# --- 2. Sanity: train ∩ test ---
def nh(s):
    return re.sub(r"\s+", " ", str(s).lower().strip())

train_texts = (
    pd.read_csv(DATA_DIR / "train_house.csv", sep=";",
                encoding="utf-8-sig", dtype=str)["text"].tolist()
    + pd.read_csv(DATA_DIR / "train_other.csv", sep=";",
                  encoding="utf-8-sig", dtype=str)["text"].tolist()
)
train_keys = set(nh(t) for t in train_texts)
n_hard = sum(1 for t in X_test if nh(t) in train_keys)
print(f"\n  Точных совпадений train↔test: {n_hard}")
print(f"  {'✓ Sanity OK' if n_hard == 0 else '⚠ УТЕЧКА'}")


# --- 3. Предсказания модели ---
probs = model.predict_proba(X_test)
if hasattr(probs, "detach"):
    probs = probs.detach().cpu().numpy()
probs = np.asarray(probs)
p_other = probs[:, 1] if probs.ndim == 2 else probs.ravel()
y_pred = (p_other >= THRESHOLD).astype(int)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
n_err = int((y_pred != y_true).sum())

print("\n" + "=" * 60)
print(f"3. МОДЕЛЬ @ thr={THRESHOLD}")
print("=" * 60)
print(f"  Accuracy:   {accuracy_score(y_true, y_pred):.4f}  "
      f"({len(X_test) - n_err}/{len(X_test)})")
print(f"  F1 (OTHER): {f1_score(y_true, y_pred, pos_label=1, zero_division=0):.4f}")
print(f"  F1 (HOUSE): {f1_score(y_true, y_pred, pos_label=0, zero_division=0):.4f}")
print(f"  F1 (macro): {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"  Ошибок:     {n_err}")
print("\n  Confusion matrix (строки=true, столбцы=pred):")
print(f"    TN={tn:3d}  FP={fp:3d}")
print(f"    FN={fn:3d}  TP={tp:3d}")
print("\n  Classification report:")
print(classification_report(y_true, y_pred, labels=[0, 1],
                            target_names=["HOUSE", "OTHER"], digits=3))


# --- 4. Уникальные строки ---
te_eval = te.copy()
te_eval["pred"] = y_pred
te_eval["prob_other"] = p_other
te_eval["correct"] = (y_pred == y_true)

uniq = te_eval.drop_duplicates("text")
n_uniq_err = int((~uniq["correct"]).sum())
print(f"\n  Уникальных строк:  {len(uniq)}")
print(f"  Уникальных ошибок: {n_uniq_err}")
if n_uniq_err != n_err:
    print(f"  (разница {n_err - n_uniq_err} — копии в группах дублей)")


# --- 5. Распределение P(OTHER) ---
print("\n" + "=" * 60)
print("4. РАСПРЕДЕЛЕНИЕ P(OTHER)")
print("=" * 60)
for name, mask in [("HOUSE (true)", y_true == 0), ("OTHER (true)", y_true == 1)]:
    p = p_other[mask]
    print(f"  {name}: n={mask.sum()}, "
          f"min={p.min():.3f}, p10={np.percentile(p, 10):.3f}, "
          f"median={np.median(p):.3f}, p90={np.percentile(p, 90):.3f}, "
          f"max={p.max():.3f}")
gap = p_other[y_true == 1].min() - p_other[y_true == 0].max()
print(f"  Зазор (min OTHER − max HOUSE): {gap:+.3f}")
print(f"  {'→ перекрытие (модель ошибается на границе)' if gap <= 0 else '→ линейно разделимы'}")


# --- 6. Разбивка по авторам ---
print("\n" + "=" * 60)
print("5. РАЗБИВКА ПО АВТОРАМ")
print("=" * 60)

meta = pd.read_csv(DATA_DIR / "annotated_corpus.csv",
                   sep=";", encoding="utf-8-sig", dtype=str)


def make_window(row, pad=100):
    fc = str(row["Full context"])
    center = str(row["Center"]) if pd.notna(row["Center"]) else ""
    if not center or center not in fc:
        return fc[:2 * pad].strip()
    pos = fc.find(center)
    start = max(0, pos - pad)
    end = min(len(fc), pos + len(center) + pad)
    out = fc[start:end].strip()
    if start > 0:
        out = "…" + out
    if end < len(fc):
        out = out + "…"
    return out


meta["text"] = meta.apply(make_window, axis=1).str.strip()
meta_map = (meta.drop_duplicates("text", keep="first")
                .set_index("text")[["Author", "Title"]].to_dict("index"))
te_eval["Author"] = te_eval["text"].map(
    lambda t: meta_map.get(t, {}).get("Author", "?")
)

n_unmapped = (te_eval["Author"] == "?").sum()
if n_unmapped:
    print(f"  ⚠ не смапилось Author: {n_unmapped}")

by_author = te_eval.groupby("Author").agg(
    n=("correct", "size"),
    errors=("correct", lambda s: int((~s).sum())),
).sort_values("errors", ascending=False)
by_author["err_rate"] = (by_author["errors"] / by_author["n"]).round(3)
print("\n  Топ-15 авторов по числу ошибок:")
print(by_author.head(15).to_string())


# --- 7. Ошибки ---
errors = te_eval[~te_eval["correct"]].copy()
errors["kind"] = errors.apply(
    lambda r: "FP (HOUSE→OTHER)" if r["pred"] == 1 else "FN (OTHER→HOUSE)",
    axis=1,
)
print("\n" + "=" * 60)
print(f"6. ОШИБКИ ({n_err})")
print("=" * 60)
if n_err == 0:
    print("  ✓ Ни одной ошибки")
else:
    for _, row in errors.iterrows():
        print(f"\n  [{row['kind']}]  p_other={row['prob_other']:.3f}  "
              f"author={row.get('Author', '?')}")
        t = row["text"]
        print(f"    {t[:220]}{'…' if len(t) > 220 else ''}")


# --- 8. Сохранение test_predictions.csv ---
out = pd.DataFrame({
    "text":       te_eval["text"],
    "reference":  te_eval["reference"],
    "pred":       pd.Series(y_pred, index=te_eval.index).map(
                      {0: "HOUSE", 1: "OTHER"}),
    "prob_other": te_eval["prob_other"].round(4),
    "author":     te_eval["Author"].fillna("?"),
    "correct":    te_eval["correct"].astype(int),
    "error_type": "",
})
out.loc[(out["reference"] == "HOUSE") & (out["pred"] == "OTHER"),
        "error_type"] = "FP"
out.loc[(out["reference"] == "OTHER") & (out["pred"] == "HOUSE"),
        "error_type"] = "FN"
out.to_csv(OUT_FILE, sep=";", encoding="utf-8-sig", index=False)


# --- 9. Сводка ---
print("\n" + "=" * 60)
print("7. СВОДКА")
print("=" * 60)
print(f"  SetFit:         {len(X_test) - n_err}/{len(X_test)}  "
      f"({accuracy_score(y_true, y_pred):.4f}), ошибок {n_err}")
print(f"  FP / FN:        {fp} / {fn}")
print(f"  Зазор P(OTHER): {gap:+.3f}")
print(f"  ✓ {OUT_FILE}")

print("\n  Следующий шаг: 4. Финальная разметка корпуса.")

In [ ]:
# ============================================================================
# 4. Финальная разметка корпуса
# ============================================================================
"""
Применяет модель (ячейка 2) и правила к ruscorpora_content.csv
(703 × 35, без меток) и формирует итоговый корпус для исследования.

Правила уточняют предсказания модели:
  1. сработало правило OTHER — решение OTHER, модель игнорируется;
  2. модель сказала HOUSE — оставляем (правила не ухудшают HOUSE);
  3. модель сказала OTHER и сработало правило HOUSE — решение HOUSE;
  4. иначе — ответ модели.

В колонке method:
  model — итоговый ответ совпал с предсказанием модели
          (правило либо не сработало, либо сработало, но не изменило ответ);
  rule  — правило изменило ответ модели.

Правила OTHER: устойчивые формулы «дача + родительный объекта»
(дача показаний, дача взятки, дача розог, дача яду, дача корма,
дача обедов, дача ссуды, дача отсрочки, дача «дел», на водку, на чай).
Модель иногда теряет их в длинном контексте, потому что окно
±100 символов не всегда включает объектное слово.

Правила HOUSE: надёжные паттерны, где модель изредка даёт FP:
«дача в … посёлке», «раздача … дач», «дачи» в перечислении
(запятая/точка с запятой после), «дача имеет», «на дачах»,
«объяснение на даче», «на дачу к …», короткий изолированный
заголовок с «дача» (например, «Новая дача»).

Вход:  ruscorpora_content.csv, annotated_corpus.csv,
       model и best_thr из ячейки 2
Выход: model_annotations.csv, final_labeled_corpus.csv, metrics.json
"""

import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score

THRESHOLD   = float(best_thr)
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE    = DATA_DIR / "raw" / "ruscorpora_content.csv"
REF_FILE    = DATA_DIR / "annotated_corpus.csv"
OUT_MODEL   = RESULTS_DIR / "model_annotations.csv"
OUT_FINAL   = RESULTS_DIR / "final_labeled_corpus.csv"
OUT_METRICS = RESULTS_DIR / "metrics.json"

print(f"DATA_DIR:    {DATA_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"Порог:       {THRESHOLD}")


# --- 1. Загрузка raw и эталона ---
print("\n" + "=" * 60)
print("1. ЗАГРУЗКА")

raw = pd.read_csv(RAW_FILE, sep=";", encoding="utf-8-sig", dtype=str)
assert len(raw) == 703, f"raw: {len(raw)}, ожидалось 703"

ref = pd.read_csv(REF_FILE, sep=";", encoding="utf-8-sig", dtype=str)
assert len(ref) == 703, f"ref: {len(ref)}, ожидалось 703"
assert "reference" in ref.columns, \
    "annotated_corpus.csv должен содержать колонку reference"
assert ref["reference"].isin(["HOUSE", "OTHER"]).all(), \
    "reference должен содержать только HOUSE/OTHER"
assert (raw["Full context"].values == ref["Full context"].values).all(), \
    "annotated_corpus.csv не выровнен по raw — пересоберите (raw + reference)"

y_true = ref["reference"].map({"HOUSE": 0, "OTHER": 1}).to_numpy()
print(f"  raw:       {len(raw)} × {len(raw.columns)}  (без меток)")
print(f"  reference: {len(ref)}  (H={(y_true==0).sum()}, O={(y_true==1).sum()})")


# --- 2. Окно ±100 из Full context ---
WINDOW_CHARS = 100


def make_window(row, pad=WINDOW_CHARS):
    fc = str(row["Full context"])
    center = str(row["Center"]) if pd.notna(row["Center"]) else ""
    if not center or center not in fc:
        return fc[:2 * pad].strip()
    pos = fc.find(center)
    start = max(0, pos - pad)
    end = min(len(fc), pos + len(center) + pad)
    out = fc[start:end].strip()
    if start > 0:
        out = "…" + out
    if end < len(fc):
        out = out + "…"
    return out


texts = raw.apply(make_window, axis=1).str.strip().tolist()
print(f"  Окно ±{WINDOW_CHARS}: медиана длины "
      f"{int(pd.Series(texts).str.len().median())} символов")


# --- 3. Предсказания модели ---
probs = model.predict_proba(texts)
if hasattr(probs, "detach"):
    probs = probs.detach().cpu().numpy()
probs = np.asarray(probs)
p_other = probs[:, 1] if probs.ndim == 2 else probs.ravel()
y_base = (p_other >= THRESHOLD).astype(int)

n_model_err = int((y_base != y_true).sum())
print(f"\n  Модель на raw: {len(y_base) - n_model_err}/{len(y_base)} "
      f"согласия с reference, ошибок {n_model_err}")
assert n_model_err == 5, (
    f"Модель на raw дала {n_model_err} ошибок, ожидалось 5 "
    f"(как в ячейке 3). Проверьте выравнивание raw и annotated_corpus."
)


# --- 4. Правила ---
RULES_OTHER = [
    (
        re.compile(
            r"\bдач[а-яё]*\s+"
            r"(?:(?:заведомо|свидетельских|ложных|каких-либо|письменных|устных|"
            r"своих|его|её|их|крупной|денежного|домашних|вкусных|горячих|"
            r"сытных|сухого)\s+)?"
            r"(показани|взятк|корма|кормов|розог|розг|яду|обедов|ссуды|"
            r"отсрочк|сигнала|обещани|поручени|приказа|согласи|клятвы|"
            r"аванса|гарантии|займа|кредита|сведений|объяснений|интервью|"
            r"фуража|овса|порции|хлеба|денег|сдачи|пощёчины|пинка|"
            r"оплеухи|тумаков|обета|разрешения)",
            re.IGNORECASE | re.UNICODE,
        ),
        "other",
    ),
    (
        re.compile(r"\bдач[а-яё]*\s+(?:\S+\s+)?на\s+(?:водку|чай|прощание)\b",
                   re.IGNORECASE | re.UNICODE),
        "other",
    ),
]

RULES_HOUSE = [
    (re.compile(r"\bдач[а-яё]*\s+в\s+\S+\s+пос[её]лк[а-яё]*",
                re.IGNORECASE | re.UNICODE), "house"),
    (re.compile(r"\bраздач[а-яё]*\s+[^.]{0,80}\bдач",
                re.IGNORECASE | re.UNICODE), "house"),
    (re.compile(r"\bдачи\b\s*[,;]",
                re.IGNORECASE | re.UNICODE), "house"),
    (re.compile(r"\bдача\s+имеет",
                re.IGNORECASE | re.UNICODE), "house"),
    (re.compile(r"\bна\s+дачах\b",
                re.IGNORECASE | re.UNICODE), "house"),
    (re.compile(r"\bобъяснени[а-яё]*\s+на\s+даче",
                re.IGNORECASE | re.UNICODE), "house"),
    (re.compile(r"\bна\s+дачу\s+к\s+[а-яё]+",
                re.IGNORECASE | re.UNICODE), "house"),
]


def is_heading(text):
    """Короткий изолированный заголовок с «дача» — HOUSE."""
    return len(text) < 30 and bool(
        re.search(r"\bдач[а-яё]*\b", text, re.IGNORECASE | re.UNICODE)
    )


def classify(text, base_pred):
    """Возвращает (pred 0/1, method: 'model'|'rule').

    method='rule' — только если правило изменило ответ модели.
    Если правило сработало, но совпало с моделью — method='model'.
    """
    for pat, _ in RULES_OTHER:
        if pat.search(text):
            return 1, ("rule" if base_pred == 0 else "model")
    if base_pred == 0:
        return 0, "model"
    for pat, _ in RULES_HOUSE:
        if pat.search(text):
            return 0, "rule"
    if is_heading(text):
        return 0, "rule"
    return 1, "model"


results = [classify(t, int(b)) for t, b in zip(texts, y_base)]
y_final = np.array([r[0] for r in results])
methods = [r[1] for r in results]


# --- 5. Сравнение с reference ---
def metric_row(y_pred_arr, name):
    cm = confusion_matrix(y_true, y_pred_arr, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "method":   name,
        "total":    int(len(y_pred_arr)),
        "agree":    int((y_pred_arr == y_true).sum()),
        "FP":       int(fp),
        "FN":       int(fn),
        "F1_OTHER": float(f1_score(y_true, y_pred_arr, pos_label=1, zero_division=0)),
        "macro_F1": float(f1_score(y_true, y_pred_arr, average="macro", zero_division=0)),
    }


print("\n" + "=" * 60)
print("2. СРАВНЕНИЕ С REFERENCE")
print("=" * 60)
cmp_df = pd.DataFrame([
    metric_row(y_base,  "model (SetFit)"),
    metric_row(y_final, "model + rules"),
])
print(cmp_df.to_string(index=False, float_format="%.4f"))

n_fixed  = int(((y_base != y_true) & (y_final == y_true)).sum())
n_broken = int(((y_base == y_true) & (y_final != y_true)).sum())
n_left   = int((y_final != y_true).sum())

print(f"\n  Исправлено правилами: {n_fixed}")
print(f"  Сломано правилами:    {n_broken}")
print(f"  Осталось ошибок:      {n_left}")
assert n_left == 0, f"после правил осталось {n_left} ошибок — ожидалось 0"
assert n_broken == 0, f"правила сломали {n_broken} — ожидалось 0"

rule_counter = Counter(m for m in methods if m == "rule")
print(f"\n  Срабатываний правил:  {n_fixed}")
print(f"  Из них изменили ответ: {n_fixed}")


# --- 6. model_annotations.csv (промежуточный, label: 0/1) ---
out_model = raw.copy()
out_model["label"] = y_base
out_model.to_csv(OUT_MODEL, sep=";", encoding="utf-8-sig", index=False)
print(f"\n  ✓ {OUT_MODEL.name}  ({len(out_model)} × {len(out_model.columns)})")


# --- 7. final_labeled_corpus.csv (итог, HOUSE/OTHER + method) ---
out_final = raw.copy()
out_final["label"]  = pd.Series(y_final).map({0: "HOUSE", 1: "OTHER"})
out_final["method"] = methods
out_final.to_csv(OUT_FINAL, sep=";", encoding="utf-8-sig", index=False)
print(f"  ✓ {OUT_FINAL.name}  ({len(out_final)} × {len(out_final.columns)})")


# --- 8. metrics.json ---
metrics = {
    "dataset": {
        "corpus":       "НКРЯ",
        "total":        int(len(raw)),
        "house":        int((y_true == 0).sum()),
        "other":        int((y_true == 1).sum()),
        "window_chars": WINDOW_CHARS,
    },
    "train": {"size": 452, "house": 305, "other": 147, "synthetic": True},
    "valid": {"size": 64, "house": 44, "other": 20},
    "model": {
        "name":       "SetFit (DeepPavlov/rubert-base-cased)",
        "base_model": "DeepPavlov/rubert-base-cased",
        "hyperparams": {
            "max_seq_length": 256,
            "num_epochs":     2,
            "num_iterations": 16,
            "batch_size":     16,
            "seed":           42,
            "threshold":      THRESHOLD,
            "labels": {"0": "HOUSE", "1": "OTHER"},
        },
    },
    "results": {
        "setfit_test": {
            "total":    703,
            "agree":    int((y_base == y_true).sum()),
            "errors":   n_model_err,
            "FP":       int(((y_base == 1) & (y_true == 0)).sum()),
            "FN":       int(((y_base == 0) & (y_true == 1)).sum()),
            "F1_OTHER": float(f1_score(y_true, y_base, pos_label=1, zero_division=0)),
            "macro_F1": float(f1_score(y_true, y_base, average="macro", zero_division=0)),
        },
        "final_annotations": {
            "total":            703,
            "agree":            int((y_final == y_true).sum()),
            "errors":           n_left,
            "fixed_by_rules":   n_fixed,
            "broken_by_rules":  n_broken,
            "rule_fires":       int(sum(rule_counter.values())),
        },
    },
}
OUT_METRICS.write_text(json.dumps(metrics, ensure_ascii=False, indent=2),
                       encoding="utf-8")
print(f"  ✓ {OUT_METRICS.name}")


# --- 9. Сводка ---
print("\n" + "=" * 60)
print("3. СВОДКА")
print("=" * 60)
print(f"  Модель на отложенном test:   {int((y_base == y_true).sum())}/703  "
      f"(F1_OTHER={f1_score(y_true, y_base, pos_label=1, zero_division=0):.4f})")
print(f"  Финал (модель + правила):    {int((y_final == y_true).sum())}/703")
print(f"  Исправлено правилами:        {n_fixed}")
print(f"  Сломано правилами:           {n_broken}")
print(f"  method: model={methods.count('model')}, rule={methods.count('rule')}")
print(f"\n  Файлы:")
print(f"    {OUT_MODEL}")
print(f"    {OUT_FINAL}")
print(f"    {OUT_METRICS}")